# Notebook 03 — Boilerplate Removal Comparison (Experiment 3)

**Research context:** Component 4 — AI-Driven Governance, Compliance and Trust Infrastructure  
**Research question:** Does removing a small set of researcher-added boilerplate phrases from the complaint narratives change how the TF-IDF + LR baseline classifies them?

**Motivation:** Error analysis of Experiment 2 found that 26 records in the largest combined group (`CG-STATE-LEASE-030`) share a standardised reporting template that includes three boilerplate phrases not present in genuine complaint narratives. These phrases may have introduced systematic vocabulary signals that the classifier learned. This experiment conservatively removes only those literal phrases and measures the effect.

**Experiment 3 design:**
- **Arm A (Baseline Rerun):** Same TF-IDF + LR pipeline and raw text as Experiment 2, using frozen Experiment 2 fold assignments.
- **Arm B (Cleaned Text):** `BoilerplateStripper → TF-IDF → LR`, same frozen fold assignments.
- No new model is trained for production use. The saved deployable model does not include this preprocessing.
- **Primary run:** `results/experiment3_boilerplate_removal/` (generated 2026-09-20).

**Caveat:** Development cross-validation evidence only. No unseen-data generalization is claimed. Results concentrated in Fold 3 due to the large group.

---

## How to use in Colab
1. Upload and unzip the repository ZIP.
2. Set `ML_ROOT` in the Setup cell to the extracted ML directory path.
3. Run all cells — no models are trained, no data is modified, no results are overwritten.

In [ ]:
# =============================================================================
# SETUP
# =============================================================================
import sys
from pathlib import Path

ML_ROOT = Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve().parent
# ML_ROOT = Path("/content/project/.../ML")  # Colab: set this manually

SRC_DIR = ML_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"ML_ROOT: {ML_ROOT}")
print(f"ML_ROOT exists: {ML_ROOT.exists()}")

In [ ]:
# Optional installation (uncomment if needed):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ML_ROOT / "requirements.txt")], check=True)

In [ ]:
# =============================================================================
# IMPORTS AND VERSIONS
# =============================================================================
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

print(f"Python:      {sys.version.split()[0]}")
print(f"NumPy:       {np.__version__}")
print(f"Pandas:      {pd.__version__}")
print(f"scikit-learn:{sklearn.__version__}")
print(f"Matplotlib:  {matplotlib.__version__}")

In [ ]:
# =============================================================================
# ARTIFACT PATHS — Primary Experiment 3 Run
# =============================================================================
DATA_CSV = ML_ROOT / "data" / "state_land_governance_confirmed_200.csv"

# Explicitly selected primary run (chronologically first, not selected by score)
EXP3_DIR  = ML_ROOT / "results" / "experiment3_boilerplate_removal"
EXP2_DIR  = ML_ROOT / "results" / "experiment2_source_text"

AUDIT_CSV        = EXP3_DIR / "preprocessing_audit.csv"
ARM_A_OOF        = EXP3_DIR / "exp3a_baseline_rerun" / "oof_predictions.csv"
ARM_B_OOF        = EXP3_DIR / "exp3b_cleaned_text"   / "oof_predictions.csv"
COMPARISON_JSON  = EXP3_DIR / "experiment3_comparison.json"
CORRECTED_SUMMARY= EXP3_DIR / "experiment3_corrected_summary.json"
EXP2_OOF         = EXP2_DIR / "out_of_fold_predictions.csv"

required = {
    "dataset":           DATA_CSV,
    "preprocessing_audit": AUDIT_CSV,
    "arm_a_oof":         ARM_A_OOF,
    "arm_b_oof":         ARM_B_OOF,
    "corrected_summary": CORRECTED_SUMMARY,
    "exp2_oof":          EXP2_OOF,
}
for name, path in required.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status}: {name} -> {path.name}")

missing = [p for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} required artifact(s) missing. "
        "Ensure the Experiment 3 primary run directory is present."
    )

# Dataset identity
dataset_hash = hashlib.sha256(DATA_CSV.read_bytes()).hexdigest()
EXPECTED_HASH = "26979f18f292f8a33a5ff960123a168f0652a5d8e11032b930b4dc7fcf070c61"
print(f"\nDataset hash match: {dataset_hash == EXPECTED_HASH}")

## 1. Boilerplate Phrases Removed

In [ ]:
# Import from repository code (no copy-paste of full implementation)
from boilerplate_transformer import BOILERPLATE_PHRASES, strip_research_boilerplate

print(f"Number of removable boilerplate phrases: {len(BOILERPLATE_PHRASES)}")
print("\nPhrases:")
for i, phrase in enumerate(BOILERPLATE_PHRASES, 1):
    print(f"  {i}. {phrase!r}")

## 2. Preprocessing Audit — Which Records Changed?

In [ ]:
audit = pd.read_csv(AUDIT_CSV)
print(f"Audit records: {len(audit)} (should be 200)")
assert len(audit) == 200

changed   = audit[audit["Was_Changed_By_Cleaning"]]
unchanged = audit[~audit["Was_Changed_By_Cleaning"]]
print(f"Records changed by boilerplate removal: {len(changed)} / {len(audit)}")
print(f"Records unchanged:                      {len(unchanged)} / {len(audit)}")

In [ ]:
# Show selected examples of changed records (first 3 only, truncated for readability)
print("Selected examples of changed records (first 3, truncated to 300 chars):")
for _, row in changed.head(3).iterrows():
    rid = row["Research_ID"]
    orig = str(row["Original_Text"])[:300]
    clean = str(row["Cleaned_Text"])[:300]
    print(f"\n  ID: {rid}")
    print(f"  Original: {orig}...")
    print(f"  Cleaned:  {clean}...")

## 3. Paired Transition Analysis (from Saved OOF Predictions)

All transition counts are recomputed from the saved OOF CSVs — they are not read from the historical comparison JSON (which contained an error in `cleaning_helped_count`).  
The corrected summary (`experiment3_corrected_summary.json`) is the authoritative reference.

In [ ]:
# Load OOF predictions and join
oof_a = pd.read_csv(ARM_A_OOF)
oof_b = pd.read_csv(ARM_B_OOF)

LABEL_ORDER = [
    "Administrative / Procedural / Integrity",
    "Lease Revenue / Payment / Enforcement",
    "Unauthorized Allocation / Transfer / Use",
    "Protected / Environmental Lease Misuse",
]

merged = oof_a[["Research_ID", "Fold", "True_Label", "Predicted_Label"]].rename(
    columns={"Predicted_Label": "Pred_A"})
merged = merged.merge(
    oof_b[["Research_ID", "Predicted_Label"]].rename(columns={"Predicted_Label": "Pred_B"}),
    on="Research_ID")
merged = merged.merge(audit[["Research_ID", "Was_Changed_By_Cleaning"]], on="Research_ID")

merged["A_Correct"] = merged["True_Label"] == merged["Pred_A"]
merged["B_Correct"] = merged["True_Label"] == merged["Pred_B"]
merged["Helped"]    = (~merged["A_Correct"]) & merged["B_Correct"]
merged["Hurt"]      = merged["A_Correct"]   & (~merged["B_Correct"])

print(f"Joined records: {len(merged)} (should be 200)")
assert len(merged) == 200

In [ ]:
# Transition table helper
def transition_table(df, label):
    n = len(df)
    a_ok     = int(df["A_Correct"].sum())
    b_ok     = int(df["B_Correct"].sum())
    helped   = int(df["Helped"].sum())
    hurt     = int(df["Hurt"].sum())
    both_ok  = int((df["A_Correct"] & df["B_Correct"]).sum())
    both_bad = int((~df["A_Correct"] & ~df["B_Correct"]).sum())
    check    = (b_ok - a_ok) == (helped - hurt)
    return pd.DataFrame([{
        "Population": label, "n": n,
        "Both Correct": both_ok, "Helped (A→B)": helped,
        "Hurt (B→A)": hurt, "Both Wrong": both_bad,
        "A Correct": a_ok, "B Correct": b_ok,
        "Arith. check": check
    }])

pop_all   = transition_table(merged,                               "All 200")
pop_chg   = transition_table(merged[merged["Was_Changed_By_Cleaning"]],  "Changed 19")
pop_unch  = transition_table(merged[~merged["Was_Changed_By_Cleaning"]], "Unchanged 181")

tables = pd.concat([pop_all, pop_chg, pop_unch], ignore_index=True).set_index("Population")
print("Paired transition counts by population:")
print(tables.to_string())
print("\n'Arith. check': (B Correct - A Correct) == (Helped - Hurt). Must be True for all rows.")

all_checks_pass = tables["Arith. check"].all()
print(f"\nAll arithmetic checks pass: {all_checks_pass}")
if not all_checks_pass:
    raise ValueError("Arithmetic identity failed — check OOF data integrity.")

In [ ]:
# IDs of records that changed outcome
helped_ids = sorted(merged[merged["Helped"]]["Research_ID"].tolist())
hurt_ids   = sorted(merged[merged["Hurt"]  ]["Research_ID"].tolist())

print(f"Records fixed by cleaning ({len(helped_ids)} total):")
print(f"  Changed-text: {[i for i in helped_ids if audit[audit['Research_ID']==i]['Was_Changed_By_Cleaning'].values[0]]}")
print(f"  Unchanged-text: {[i for i in helped_ids if not audit[audit['Research_ID']==i]['Was_Changed_By_Cleaning'].values[0]]}")
print(f"\nRecords newly wrong after cleaning ({len(hurt_ids)} total):")
print(f"  {hurt_ids if hurt_ids else '(none)'}")

## 4. Overall Metrics Comparison

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "Accuracy":    round(accuracy_score(y_true, y_pred), 4),
        "Macro-F1":    round(f1_score(y_true, y_pred, average="macro", labels=LABEL_ORDER, zero_division=0), 4),
        "Weighted-F1": round(f1_score(y_true, y_pred, average="weighted", labels=LABEL_ORDER, zero_division=0), 4),
        "Macro-P":     round(precision_score(y_true, y_pred, average="macro", labels=LABEL_ORDER, zero_division=0), 4),
        "Macro-R":     round(recall_score(y_true, y_pred, average="macro", labels=LABEL_ORDER, zero_division=0), 4),
        "Misclassified": int(sum(a!=b for a,b in zip(y_true, y_pred))),
    }

a_m = compute_metrics(oof_a["True_Label"].tolist(), oof_a["Predicted_Label"].tolist())
b_m = compute_metrics(oof_b["True_Label"].tolist(), oof_b["Predicted_Label"].tolist())

comparison_df = pd.DataFrame({"Arm A (Baseline)": a_m, "Arm B (Cleaned)": b_m}).T
comparison_df["Delta (B-A)"] = comparison_df["Arm B (Cleaned)"] - comparison_df["Arm A (Baseline)"]
print("Overall metric comparison (n=200, development CV):")
print(comparison_df.round(4).to_string())

In [ ]:
# Per-class metrics
def per_class_table(y_true, y_pred, arm_name):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    rows = []
    for i, lbl in enumerate(LABEL_ORDER):
        tp = cm[i,i]; fp = int(cm[:,i].sum()-tp); fn = int(cm[i,:].sum()-tp)
        tn = int(cm.sum()-tp-fp-fn)
        p  = tp/(tp+fp) if tp+fp>0 else 0
        r  = tp/(tp+fn) if tp+fn>0 else 0
        f1 = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0
        fpr= fp/(fp+tn) if (fp+tn)>0 else 0
        rows.append({"Class": lbl[:35], "Arm": arm_name,
                     "Precision": round(p,4), "Recall": round(r,4),
                     "F1": round(f1,4), "FPR": round(fpr,4)})
    return pd.DataFrame(rows)

a_pc = per_class_table(oof_a["True_Label"].tolist(), oof_a["Predicted_Label"].tolist(), "Arm A")
b_pc = per_class_table(oof_b["True_Label"].tolist(), oof_b["Predicted_Label"].tolist(), "Arm B")
pc_df = pd.concat([a_pc, b_pc]).pivot_table(index="Class", columns="Arm", values=["Precision","Recall","F1","FPR"])
print("Per-class metrics comparison:")
print("(F1 values are per-class F1 scores, not macro-averaged)")
print(pc_df.round(4).to_string())

In [ ]:
# Side-by-side confusion matrices
cm_a = confusion_matrix(oof_a["True_Label"], oof_a["Predicted_Label"], labels=LABEL_ORDER)
cm_b = confusion_matrix(oof_b["True_Label"], oof_b["Predicted_Label"], labels=LABEL_ORDER)
short_labels = [l.split("/")[0].strip()[:18] for l in LABEL_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, cm, title in zip(axes, [cm_a, cm_b],
                          ["Arm A — Baseline (n=200)", "Arm B — Cleaned (n=200)"]):
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_labels)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.suptitle("Experiment 3 — Confusion Matrices (Development CV, Frozen Exp.2 Folds)", y=1.02)
plt.tight_layout()
plt.show()

## 5. Arm A vs Experiment 2 Identity

In [ ]:
exp2_oof = pd.read_csv(EXP2_OOF)

cmp = exp2_oof[["Research_ID", "Predicted_Label", "Fold", "True_Label"]].merge(
    oof_a[["Research_ID", "Predicted_Label", "Fold", "True_Label"]].rename(
        columns={"Predicted_Label": "ArmA_Pred", "Fold": "ArmA_Fold", "True_Label": "ArmA_True"}),
    on="Research_ID")

id_pred  = int((cmp["Predicted_Label"] == cmp["ArmA_Pred"]).sum())
id_fold  = int((cmp["Fold"] == cmp["ArmA_Fold"]).sum())
id_true  = int((cmp["True_Label"] == cmp["ArmA_True"]).sum())

print(f"Arm A vs Experiment 2 (by Research_ID, n=200):")
print(f"  Identical predicted labels:    {id_pred}/200")
print(f"  Identical fold assignments:    {id_fold}/200")
print(f"  Identical true labels:         {id_true}/200")

# Saved probabilities if available
prob_cols = [c for c in exp2_oof.columns if c.startswith("prob_")]
if prob_cols:
    a_prob_cols = [c for c in oof_a.columns if c.startswith("prob_")]
    e2_probs = exp2_oof.sort_values("Research_ID")[prob_cols].values
    a_probs  = oof_a.sort_values("Research_ID")[a_prob_cols].values
    max_diff = float(np.abs(e2_probs - a_probs).max())
    print(f"  Max probability difference:    {max_diff:.6f}")
else:
    print("  Saved probabilities: not available in OOF CSV")

## 6. Subgroup Analysis — CG-STATE-LEASE-030 and Fold 3

In [ ]:
# CG-STATE-LEASE-030: the 26-record largest group, all in Fold 3
cg_group = "CG-STATE-LEASE-030"

if "Combined_Group" in merged.columns:
    cg_df = merged[merged["Combined_Group"] == cg_group]
else:
    # Fallback: use audit Combined_Group
    if "Combined_Group" in audit.columns:
        merged_cg = merged.merge(audit[["Research_ID", "Combined_Group"]], on="Research_ID")
        cg_df = merged_cg[merged_cg["Combined_Group"] == cg_group]
    else:
        # Use Exp2 OOF to get groups
        grp_map = dict(zip(exp2_oof["Research_ID"], exp2_oof.get("Combined_Group", pd.Series())))
        merged["Combined_Group"] = merged["Research_ID"].map(grp_map)
        cg_df = merged[merged["Combined_Group"] == cg_group]

print(f"Group {cg_group}: {len(cg_df)} records")
tables_cg = pd.concat([
    transition_table(cg_df, f"CG-030 (all {len(cg_df)})"),
    transition_table(cg_df[cg_df["Was_Changed_By_Cleaning"]], f"CG-030 changed ({cg_df['Was_Changed_By_Cleaning'].sum()})"),
    transition_table(cg_df[~cg_df["Was_Changed_By_Cleaning"]], f"CG-030 unchanged ({(~cg_df['Was_Changed_By_Cleaning']).sum()})"),
], ignore_index=True).set_index("Population")
print(tables_cg.to_string())

In [ ]:
# Fold 3 vs remaining folds
fold3   = merged[merged["Fold"] == 3]
others  = merged[merged["Fold"] != 3]

fold_tables = pd.concat([
    transition_table(fold3,  f"Fold 3 (n={len(fold3)})"),
    transition_table(others, f"Folds 1,2,4,5 (n={len(others)})"),
], ignore_index=True).set_index("Population")
print("Fold breakdown:")
print(fold_tables.to_string())
print(
    "\nNote: All 8 improvements within changed-text records are in Fold 3. "
    "This reflects the concentration of boilerplate-affected records in that fold."
)

## 7. Corrected Summary Reference

In [ ]:
with open(CORRECTED_SUMMARY, "r", encoding="utf-8") as f:
    corr = json.load(f)

print("Corrected summary source paths:")
for k, v in corr["source_artifact_paths"].items():
    print(f"  {k}: {v}")

print(f"\nError in original JSON:")
err = corr["error_in_original_json"]
print(f"  Field:           {err['field']}")
print(f"  Erroneous value: {err['erroneous_value']}")
print(f"  Correct value:   {err['correct_value']}")
print(f"  Cause:           {err['cause']}")

print(f"\nNote on WEB-157: {corr['web_157_note']}")

## 8. Interpretation and Limitations

- The metric improvement (Acc +0.045, Macro-F1 +0.044) is concentrated in **Fold 3**, where all 26 records of the largest boilerplate-template group are tested together. This does not generalise to all record types.
- **WEB-157** improved in Arm B despite not containing any boilerplate phrases. Its prediction changed when training-set vocabulary was altered by cleaning other records in the same fold's training partition. The mechanism was not directly measured.
- **The saved deployable model does not include boilerplate removal.** This is a preprocessing experiment only.
- The original `experiment3_comparison.json` contains an erroneous `cleaning_helped_count = 9` in the `changed_records_analysis` section. The correct value is **8** (for the 19 changed records). The corrected summary is the authoritative reference.
- Results are exploratory development evidence. No unseen-data generalization or statistical significance is claimed.

### Optional: Reproduce Experiment 3

To reproduce from scratch (requires full environment and Exp. 2 OOF artifacts):
```bash
cd <ML_ROOT>
python src/evaluate_experiment3.py
```
A new numbered run directory will be created automatically (existing runs are preserved).